# Bird Species Observation Analysis
## Phase 1 : Data Cleaning & Preprocessing

**Goal :** Read all Excel sheets → Clean each habitat separately → Merge → Export CSV

**Files needed in the same folder as this notebook:**
- `Bird_Monitoring_Data_FOREST.XLSX`
- `Bird_Monitoring_Data_GRASSLAND.XLSX`

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

---
## Step 2 — Read FOREST Excel (all 11 sheets)

In [ ]:
# pd.read_excel with sheet_name=None reads ALL sheets at once
# It returns a dictionary  {sheet_name : dataframe}

forest_sheets = pd.read_excel('Bird_Monitoring_Data_FOREST.XLSX', sheet_name=None)

print('FOREST sheets found:', list(forest_sheets.keys()))

In [ ]:
# Loop through the dictionary and stack (vertically concat) non-empty sheets

forest_frames = []   # empty list to collect each sheet's dataframe

for sheet_name, df_sheet in forest_sheets.items():

    if df_sheet.empty:
        print(f'  SKIP  : {sheet_name} — 0 rows')
        continue

    df_sheet['Source_Sheet']   = sheet_name    # tag which sheet it came from
    df_sheet['Habitat_Source'] = 'Forest'      # tag the habitat
    forest_frames.append(df_sheet)
    print(f'  LOADED: {sheet_name} — {len(df_sheet)} rows')

# Stack all sheets into one dataframe
df_forest = pd.concat(forest_frames, ignore_index=True)

print(f'\ndf_forest shape: {df_forest.shape}')

---
## Step 3 — Read GRASSLAND Excel (all 11 sheets)

In [ ]:
grassland_sheets = pd.read_excel('Bird_Monitoring_Data_GRASSLAND.XLSX', sheet_name=None)

print('GRASSLAND sheets found:', list(grassland_sheets.keys()))

In [ ]:
grassland_frames = []

for sheet_name, df_sheet in grassland_sheets.items():

    if df_sheet.empty:
        print(f'  SKIP  : {sheet_name} — 0 rows (empty in source file)')
        continue

    df_sheet['Source_Sheet']   = sheet_name
    df_sheet['Habitat_Source'] = 'Grassland'
    grassland_frames.append(df_sheet)
    print(f'  LOADED: {sheet_name} — {len(df_sheet)} rows')

df_grass = pd.concat(grassland_frames, ignore_index=True)

print(f'\ndf_grass shape: {df_grass.shape}')

---
## Step 4 — Fix Schema Differences Between Forest and Grassland

The two files have slightly different column names:

| Column | Forest file | Grassland file |
|---|---|---|
| Taxon code | `NPSTaxonCode` | `TaxonCode` |
| Previously observed | missing | `Previously_Obs` |
| Site name | `Site_Name` | missing |

In [ ]:
# Fix Forest — rename NPSTaxonCode to TaxonCode
if 'NPSTaxonCode' in df_forest.columns:
    df_forest = df_forest.rename(columns={'NPSTaxonCode': 'TaxonCode'})
    print('Forest: renamed NPSTaxonCode -> TaxonCode')

# Fix Forest — add Previously_Obs column (does not exist in Forest)
df_forest['Previously_Obs'] = np.nan
print('Forest: added Previously_Obs column as NaN')

# Fix Grassland — add Site_Name column (does not exist in Grassland)
if 'Site_Name' not in df_grass.columns:
    df_grass['Site_Name'] = np.nan
    print('Grassland: added Site_Name column as NaN')

In [ ]:
# Define the final column order that both dataframes must follow

FINAL_COLUMNS = [
    'Admin_Unit_Code', 'Sub_Unit_Code', 'Site_Name', 'Plot_Name',
    'Location_Type', 'Year', 'Date', 'Start_Time', 'End_Time',
    'Observer', 'Visit', 'Interval_Length', 'ID_Method', 'Distance',
    'Flyover_Observed', 'Sex', 'Common_Name', 'Scientific_Name',
    'AcceptedTSN', 'TaxonCode', 'AOU_Code', 'PIF_Watchlist_Status',
    'Regional_Stewardship_Status', 'Temperature', 'Humidity', 'Sky',
    'Wind', 'Disturbance', 'Previously_Obs', 'Initial_Three_Min_Cnt',
    'Habitat_Source', 'Source_Sheet'
]

df_forest = df_forest.reindex(columns=FINAL_COLUMNS)
df_grass  = df_grass.reindex(columns=FINAL_COLUMNS)

print('Forest columns  :', df_forest.shape[1])
print('Grassland columns:', df_grass.shape[1])
print('Both match:', df_forest.shape[1] == df_grass.shape[1])

---
## Step 5 — Clean FOREST

> Cleaning is done on each habitat **separately** before merging.  
> This ensures the IQR outlier fence for temperature is calculated  
> from Forest data only — not influenced by Grassland values.

In [ ]:
# ── 5a. Date column ──────────────────────────────────────────────────────
df_forest['Date'] = pd.to_datetime(df_forest['Date'], errors='coerce')

# Derive Month, Month Name, Season from Date
df_forest['Month']      = df_forest['Date'].dt.month
df_forest['Month_Name'] = df_forest['Date'].dt.strftime('%b')

season_map = {
    1:'Winter', 2:'Winter', 3:'Spring', 4:'Spring',  5:'Spring',
    6:'Summer', 7:'Summer', 8:'Summer', 9:'Autumn', 10:'Autumn',
    11:'Autumn', 12:'Winter'
}
df_forest['Season'] = df_forest['Month'].map(season_map)

print('Date range:', df_forest['Date'].min().date(), 'to', df_forest['Date'].max().date())

In [ ]:
# ── 5b. Time columns ─────────────────────────────────────────────────────
# Parse Start_Time and End_Time strings to time objects
# Use pd.to_datetime(...).dt.time — this stores only the time portion

df_forest['Start_Time'] = pd.to_datetime(
    df_forest['Start_Time'].astype(str).str.strip(),
    format='%H:%M:%S', errors='coerce'
).dt.time

df_forest['End_Time'] = pd.to_datetime(
    df_forest['End_Time'].astype(str).str.strip(),
    format='%H:%M:%S', errors='coerce'
).dt.time

# Extract hour number from Start_Time for analysis (e.g. 6, 7, 8 ...)
df_forest['Start_Hour'] = df_forest['Start_Time'].apply(
    lambda t: t.hour if (t is not None and pd.notna(t)) else np.nan
)

print('Sample Start_Time values:', df_forest['Start_Time'].dropna().head(3).tolist())
print('Sample Start_Hour values:', df_forest['Start_Hour'].dropna().head(3).tolist())

In [ ]:
# ── 5c. Temperature — convert and cap outliers using IQR ─────────────────
df_forest['Temperature'] = pd.to_numeric(df_forest['Temperature'], errors='coerce')

Q1 = df_forest['Temperature'].quantile(0.25)
Q3 = df_forest['Temperature'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = round(Q1 - 1.5 * IQR, 1)
upper_fence = round(Q3 + 1.5 * IQR, 1)

# Count how many rows fall outside the fence
outliers = df_forest['Temperature'].between(lower_fence, upper_fence) == False
print(f'Forest Temperature fence : [{lower_fence}, {upper_fence}]')
print(f'Outliers capped          : {outliers.sum()} rows')

# Cap the outliers (don't drop them)
df_forest['Temperature'] = df_forest['Temperature'].clip(
    lower=lower_fence, upper=upper_fence
).round(1)

In [ ]:
# ── 5d. Humidity — fill missing values with median per Admin Unit ─────────
df_forest['Humidity'] = pd.to_numeric(df_forest['Humidity'], errors='coerce')

missing_before = df_forest['Humidity'].isna().sum()

df_forest['Humidity'] = df_forest.groupby('Admin_Unit_Code')['Humidity'].transform(
    lambda x: x.fillna(x.median())
).round(1)

print(f'Humidity nulls filled: {missing_before}')

In [ ]:
# ── 5e. Sex — fill blank values ───────────────────────────────────────────
df_forest['Sex'] = df_forest['Sex'].fillna('Undetermined').astype(str).str.strip()

print('Sex values:', df_forest['Sex'].unique())

In [ ]:
# ── 5f. Boolean columns — normalise True/False strings ───────────────────
# CSV and Excel can store booleans as True/False, 1/0, or strings
# .map() handles all cases safely

bool_map = {True: True, False: False, 'True': True, 'False': False, 1: True, 0: False}

for col in ['Flyover_Observed', 'PIF_Watchlist_Status',
            'Regional_Stewardship_Status', 'Initial_Three_Min_Cnt', 'Previously_Obs']:
    df_forest[col] = df_forest[col].map(bool_map)

print('Boolean columns normalised')

In [ ]:
# ── 5g. Derived / enrichment columns ─────────────────────────────────────

# Sky — standardise free text values
sky_map = {
    'Clear or Few Clouds': 'Clear',
    'Partly Cloudy'      : 'Partly Cloudy',
    'Cloudy/Overcast'    : 'Overcast',
    'Mist/Drizzle'       : 'Mist/Drizzle',
    'Fog'                : 'Fog',
    'Rain'               : 'Rain'
}
df_forest['Sky_Clean'] = df_forest['Sky'].astype(str).str.strip().map(sky_map).fillna('Other')

# Distance — add rank number for sorting
df_forest['Distance_Clean'] = df_forest['Distance'].astype(str).str.strip()
df_forest['Distance_Rank']  = df_forest['Distance_Clean'].map(
    {'<= 50 Meters': 1, '50 - 100 Meters': 2, '> 100 Meters': 3}
)

# Wind — category label
def classify_wind(w):
    if pd.isna(w) or str(w).strip() == '': return 'Unknown'
    w = str(w).lower()
    if 'calm'  in w: return 'Calm'
    if '1-3'   in w: return 'Light Air'
    if '4-7'   in w: return 'Light Breeze'
    if '8-12'  in w: return 'Gentle Breeze'
    return 'Strong'

df_forest['Wind_Category'] = df_forest['Wind'].apply(classify_wind)

# Disturbance — severity label
def classify_disturbance(d):
    if pd.isna(d) or str(d).strip() == '': return 'Unknown'
    d = str(d).lower()
    if 'no effect' in d: return 'None'
    if 'slight'    in d: return 'Slight'
    if 'moderate'  in d: return 'Moderate'
    if 'major'     in d or 'high' in d: return 'High'
    return 'Other'

df_forest['Disturbance_Level'] = df_forest['Disturbance'].apply(classify_disturbance)

# Interval midpoint — numeric representation of observation window
df_forest['Interval_Mid_Min'] = df_forest['Interval_Length'].astype(str).str.strip().map(
    {'0-2.5 min': 1.25, '2.5 - 5 min': 3.75, '5 - 7.5 min': 6.25, '7.5 - 10 min': 8.75}
)

# At Risk flag — True if on PIF watchlist OR regional priority
df_forest['At_Risk'] = (
    (df_forest['PIF_Watchlist_Status']        == True) |
    (df_forest['Regional_Stewardship_Status'] == True)
)

print('Forest enrichment done')
print('Forest final shape:', df_forest.shape)

---
## Step 6 — Clean GRASSLAND

> Same steps as Forest but using Grassland data only.  
> The IQR fence will be different because Grassland has different temperature ranges.

In [ ]:
# ── 6a. Date ──────────────────────────────────────────────────────────────
df_grass['Date']      = pd.to_datetime(df_grass['Date'], errors='coerce')
df_grass['Month']     = df_grass['Date'].dt.month
df_grass['Month_Name']= df_grass['Date'].dt.strftime('%b')
df_grass['Season']    = df_grass['Month'].map(season_map)

print('Date range:', df_grass['Date'].min().date(), 'to', df_grass['Date'].max().date())

In [ ]:
# ── 6b. Time ──────────────────────────────────────────────────────────────
df_grass['Start_Time'] = pd.to_datetime(
    df_grass['Start_Time'].astype(str).str.strip(),
    format='%H:%M:%S', errors='coerce'
).dt.time

df_grass['End_Time'] = pd.to_datetime(
    df_grass['End_Time'].astype(str).str.strip(),
    format='%H:%M:%S', errors='coerce'
).dt.time

df_grass['Start_Hour'] = df_grass['Start_Time'].apply(
    lambda t: t.hour if (t is not None and pd.notna(t)) else np.nan
)

print('Sample Start_Hour values:', df_grass['Start_Hour'].dropna().head(3).tolist())

In [ ]:
# ── 6c. Temperature — IQR cap on Grassland rows only ─────────────────────
df_grass['Temperature'] = pd.to_numeric(df_grass['Temperature'], errors='coerce')

Q1 = df_grass['Temperature'].quantile(0.25)
Q3 = df_grass['Temperature'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = round(Q1 - 1.5 * IQR, 1)
upper_fence = round(Q3 + 1.5 * IQR, 1)

outliers = df_grass['Temperature'].between(lower_fence, upper_fence) == False
print(f'Grassland Temperature fence : [{lower_fence}, {upper_fence}]')
print(f'Outliers capped             : {outliers.sum()} rows')

df_grass['Temperature'] = df_grass['Temperature'].clip(
    lower=lower_fence, upper=upper_fence
).round(1)

In [ ]:
# ── 6d. Humidity ──────────────────────────────────────────────────────────
df_grass['Humidity'] = pd.to_numeric(df_grass['Humidity'], errors='coerce')
missing_before = df_grass['Humidity'].isna().sum()
df_grass['Humidity'] = df_grass.groupby('Admin_Unit_Code')['Humidity'].transform(
    lambda x: x.fillna(x.median())
).round(1)
print(f'Humidity nulls filled: {missing_before}')

In [ ]:
# ── 6e. Sex ───────────────────────────────────────────────────────────────
df_grass['Sex'] = df_grass['Sex'].fillna('Undetermined').astype(str).str.strip()

In [ ]:
# ── 6f. Boolean columns ───────────────────────────────────────────────────
for col in ['Flyover_Observed', 'PIF_Watchlist_Status',
            'Regional_Stewardship_Status', 'Initial_Three_Min_Cnt', 'Previously_Obs']:
    df_grass[col] = df_grass[col].map(bool_map)

print('Boolean columns normalised')

In [ ]:
# ── 6g. Derived / enrichment columns ─────────────────────────────────────
df_grass['Sky_Clean']        = df_grass['Sky'].astype(str).str.strip().map(sky_map).fillna('Other')
df_grass['Distance_Clean']   = df_grass['Distance'].astype(str).str.strip()
df_grass['Distance_Rank']    = df_grass['Distance_Clean'].map(
    {'<= 50 Meters': 1, '50 - 100 Meters': 2, '> 100 Meters': 3}
)
df_grass['Wind_Category']     = df_grass['Wind'].apply(classify_wind)
df_grass['Disturbance_Level'] = df_grass['Disturbance'].apply(classify_disturbance)
df_grass['Interval_Mid_Min']  = df_grass['Interval_Length'].astype(str).str.strip().map(
    {'0-2.5 min': 1.25, '2.5 - 5 min': 3.75, '5 - 7.5 min': 6.25, '7.5 - 10 min': 8.75}
)
df_grass['At_Risk'] = (
    (df_grass['PIF_Watchlist_Status']        == True) |
    (df_grass['Regional_Stewardship_Status'] == True)
)

print('Grassland enrichment done')
print('Grassland final shape:', df_grass.shape)

---
## Step 7 — Final Stack : Merge Forest + Grassland

In [ ]:
print('Before merge:')
print(f'  df_forest : {df_forest.shape[0]:>6} rows x {df_forest.shape[1]} cols')
print(f'  df_grass  : {df_grass.shape[0]:>6} rows x {df_grass.shape[1]} cols')
print()

# Vertical stack — rows are appended, no key-based join
df = pd.concat([df_forest, df_grass], ignore_index=True)

print('After merge:')
print(f'  df        : {df.shape[0]:>6} rows x {df.shape[1]} cols')
print()
print('Rows per habitat:')
print(df['Habitat_Source'].value_counts().to_string())

---
## Step 8 — Quality Check

In [ ]:
print('=' * 45)
print('QUALITY REPORT')
print('=' * 45)
print(f'Total rows              : {len(df):,}')
print(f'Total columns           : {df.shape[1]}')
print(f'Date range              : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Admin units             : {sorted(df["Admin_Unit_Code"].dropna().unique())}')
print(f'Unique species          : {df["Common_Name"].nunique()}')
print(f'PIF Watchlist species   : {df[df["PIF_Watchlist_Status"]==True]["Common_Name"].nunique()}')
print(f'At-Risk species         : {df[df["At_Risk"]==True]["Common_Name"].nunique()}')
print(f'Peak observation month  : {df["Month_Name"].value_counts().idxmax()}')
print(f'Peak observation hour   : {int(df["Start_Hour"].dropna().astype(int).value_counts().idxmax())}:00')
print()
print('Remaining null counts (columns with nulls only):')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())

---
## Step 9 — Export to CSV

In [ ]:
df.to_csv('bird_observations_clean.csv', index=False)

# Verify the exported file
verify = pd.read_csv('bird_observations_clean.csv', nrows=0)

print(f'File exported  : bird_observations_clean.csv')
print(f'Columns in CSV : {len(verify.columns)}  (expected 43)')
print(f'Column names   : {list(verify.columns)}')
print()
df.head(3)